<h1><b><i>SCRIPT PRINCIPAL</b></i></h1>
<h2><b><i>Importação de Bibliotécas</b></i></h2>

In [9]:
import sys
sys.path.append(r'C:\pylibs')

import pandas as pd
import numpy as np
import openpyxl

In [10]:
import pandas as pd
import openpyxl
import numpy as np
import os

In [11]:
# ==============================================================================
# 1. SETUP INICIAL E CARREGAMENTO
#    (Melhor prática: definir constantes e carregar o arquivo de forma segura)
# ==============================================================================

# Definição do caminho do arquivo em uma variável para fácil manutenção
FILE_PATH = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21 - À EXECUTAR (2026 DIANTE).xlsx'
OUTPUT_DIR = 'Dados Gerados'

# Verifica se o arquivo existe antes de tentar carregá-lo
if not os.path.exists(FILE_PATH):
    print(f"ERRO: O arquivo '{FILE_PATH}' não foi encontrado.")
    # Encerra o script ou lança uma exceção se o arquivo principal não existir
    exit()

# Cria a pasta de saída se ela não existir
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Arquivo '{FILE_PATH}' encontrado. Iniciando processamento...")

Arquivo 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21 - À EXECUTAR (2026 DIANTE).xlsx' encontrado. Iniciando processamento...


<h1><b><i>'À EXECUTAR (2026 DIANTE)'</b></i></h1>

In [12]:
# %% =====================================================================
# LEITURA INTELIGENTE DA PLANILHA "À EXECUTAR (2026 DIANTE)"
# ========================================================================

import pandas as pd
import numpy as np
import os
from collections import defaultdict

def expandir_mescladas_para_cabecalho(df_cab):
    """Expande horizontalmente (ffill) para simular células mescladas."""
    return df_cab.ffill(axis=1)

def montar_cabecalho_hierarquico(df_cab, n_niveis=3):
    """
    Recebe um DataFrame com N linhas de cabeçalho (já com ffill)
    e retorna uma lista de nomes de colunas concatenando os níveis.
    """
    lixo = {"#REF!", "TOTAL", "SOMA", 0, "0", None, ""}
    df_cab = df_cab.replace(list(lixo), pd.NA)
    
    colunas = []
    for col in range(df_cab.shape[1]):
        partes = []
        for row in range(df_cab.shape[0]):
            valor = df_cab.iat[row, col]
            if pd.isna(valor):
                continue
            valor = str(valor).strip()
            if not valor or valor.startswith(("=", "#")):
                continue
            if valor.upper() in ("TOTAL", "SOMA"):
                continue
            try:
                float(valor.replace(",", "."))
                continue
            except:
                pass
            partes.append(valor)
        colunas.append(" - ".join(partes) if partes else "")
    
    # Remove duplicatas adicionando sufixo __n
    contador = defaultdict(int)
    novas = []
    for nome in colunas:
        contador[nome] += 1
        if contador[nome] == 1:
            novas.append(nome)
        else:
            novas.append(f"{nome}__{contador[nome]}")
    return novas

# ========================================================================
# 1. DETECTAR A ÚLTIMA COLUNA COM DADOS NO CABEÇALHO
# ========================================================================

file_path = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx'  # ajuste se necessário
sheet_name = 'À EXECUTAR (2026 DIANTE)'

# Lê as 3 primeiras linhas (cabeçalho) para identificar a última coluna não vazia
cabecalho_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, nrows=3)

# Encontra o índice da última coluna que contém algum valor não nulo
ultima_coluna_idx = cabecalho_raw.columns[cabecalho_raw.notna().any(axis=0)].max() + 1
if pd.isna(ultima_coluna_idx):
    ultima_coluna_idx = 0

num_colunas = int(ultima_coluna_idx)
print(f"🔍 Última coluna com dados: {num_colunas}")

# ========================================================================
# 2. LER CABEÇALHO E DADOS COM O MESMO NÚMERO DE COLUNAS
# ========================================================================

cabecalho_raw = pd.read_excel(
    file_path, sheet_name=sheet_name, header=None, nrows=3, usecols=range(num_colunas)
)

# Expande horizontalmente (simula mesclas)
cabecalho_expandido = expandir_mescladas_para_cabecalho(cabecalho_raw)

# Gera os nomes das colunas (3 níveis)
novos_nomes = montar_cabecalho_hierarquico(cabecalho_expandido, n_niveis=3)

# Lê os dados pulando as 3 primeiras linhas
df_a_executar = pd.read_excel(
    file_path, sheet_name=sheet_name, header=None, skiprows=3, usecols=range(num_colunas)
)

# Atribui os nomes gerados
df_a_executar.columns = novos_nomes

# Remove linhas totalmente vazias
df_a_executar = df_a_executar.dropna(how='all')

print(f"✅ Planilha 'À EXECUTAR' carregada com {df_a_executar.shape[0]} linhas e {df_a_executar.shape[1]} colunas.")

# ========================================================================
# 3. IDENTIFICAR COLUNAS DE IDENTIFICAÇÃO E PREENCHER PARA BAIXO
# ========================================================================

KEYWORDS_ID = [
    "SETOR", "UF", "ESTADO/LOTE", "BR", "EMPREENDIMENTO", "PROPONENTE",
    "EXECUTOR", "ESTRUTURADOR", "OBSERVAÇÕES", "SITUAÇÃO GERAL"
]

id_cols = []
for col in df_a_executar.columns:
    col_upper = col.upper()
    if any(kw in col_upper for kw in KEYWORDS_ID):
        id_cols.append(col)

print("Colunas identificadas como ID:")
for c in id_cols:
    print(f"  - {c}")

# Aplica ffill apenas nessas colunas
if id_cols:
    df_a_executar[id_cols] = df_a_executar[id_cols].ffill()
    print("✅ Forward fill aplicado nas colunas de identificação.")

# ========================================================================
# 4. REMOVER LINHAS COM "TOTAL" OU "SOMA" NAS COLUNAS DE MÉTRICA
# ========================================================================

metric_cols = [c for c in df_a_executar.columns if c not in id_cols]

if metric_cols:
    mask_total = df_a_executar[metric_cols].astype(str).apply(
        lambda row: row.str.contains('TOTAL|SOMA', case=False, na=False).any(), axis=1
    )
    df_a_executar = df_a_executar[~mask_total].reset_index(drop=True)

print(f"✅ Linhas com TOTAL/SOMA removidas. Restam {df_a_executar.shape[0]} linhas.")

# ========================================================================
# 5. MELT (FORMATO LONGO) E SEPARAÇÃO DOS NÍVEIS
# ========================================================================

df_long = df_a_executar.melt(
    id_vars=id_cols,
    value_vars=metric_cols,
    var_name="cabecalho_completo",
    value_name="Valor"
)

# Remove linhas com Valor vazio ou zero (ajuste conforme necessário)
df_long = df_long.dropna(subset=["Valor"])
df_long = df_long[df_long["Valor"] != 0]

print(f"✅ Melt aplicado: {df_long.shape[0]} linhas.")

# ========================================================================
# 6. SEPARAR O CABEÇALHO EM NÍVEIS (Tipo, Período, Métrica)
# ========================================================================

# Divide pelo separador " - "
split_headers = df_long["cabecalho_completo"].str.split(" - ", expand=True)
num_niveis = split_headers.shape[1]

# Nomeia os níveis dinamicamente
if num_niveis >= 3:
    split_headers.columns = ["Tipo", "Período", "Metrica"]
elif num_niveis == 2:
    split_headers.columns = ["Tipo", "Metrica"]
    # Se não houver período, preenche com "Geral"
    df_long["Período"] = "Geral"
else:
    split_headers.columns = ["Metrica"]
    df_long["Tipo"] = "Geral"
    df_long["Período"] = "Geral"

# Concatena com o DataFrame longo
df_long = pd.concat([df_long, split_headers], axis=1)
df_long.drop(columns=["cabecalho_completo"], inplace=True)

# ========================================================================
# 7. PIVOTAR PARA TER AS MÉTRICAS COMO COLUNAS
# ========================================================================

# Define as colunas que serão usadas como índice: ID + Tipo + Período
index_cols = id_cols + ["Tipo", "Período"]

# Pivot: cada métrica vira uma coluna
df_pivot = df_long.pivot_table(
    index=index_cols,
    columns="Metrica",
    values="Valor",
    aggfunc="first"  # ou 'sum' se houver múltiplos valores para a mesma combinação
).reset_index()

# Renomeia as colunas para remover o nome da métrica (opcional)
df_pivot.columns = [col if col == '' else col for col in df_pivot.columns]

# Limpa nomes das colunas (remove espaços extras)
df_pivot.columns = [str(col).strip() for col in df_pivot.columns]

print(f"✅ Pivot concluído: {df_pivot.shape[0]} linhas x {df_pivot.shape[1]} colunas.")

# ========================================================================
# 8. SALVAR O RESULTADO
# ========================================================================

# Garante que a pasta existe
os.makedirs('Dados Gerados', exist_ok=True)

# Salva
df_pivot.to_excel(
    'Dados Gerados/À EXECUTAR (2026 DIANTE).xlsx',
    index=False,
    sheet_name='À EXECUTAR (2026 DIANTE)'
)

print("✅ Arquivo salvo em 'Dados Gerados/À EXECUTAR (2026 DIANTE).xlsx'")

🔍 Última coluna com dados: 105


c:\Users\carlos.ramos\OneDrive - Presidência da República\Área de Trabalho\Planilhas\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


✅ Planilha 'À EXECUTAR' carregada com 1938 linhas e 105 colunas.
Colunas identificadas como ID:
  - SETOR
  - UF
  - PLANILHA DE MONITORAMENTO - ESTADO/LOTE
  - PLANILHA DE MONITORAMENTO - BR
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - EMPREENDIMENTO
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - PROPONENTE
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - EXECUTOR             (Grupo Controlador)
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - ESTRUTURADOR DO PROJETO
  - PLANILHA DE MONITORAMENTO - DADOS CONTRATUAIS - OBSERVAÇÕES
✅ Forward fill aplicado nas colunas de identificação.
✅ Linhas com TOTAL/SOMA removidas. Restam 1814 linhas.
✅ Melt aplicado: 20206 linhas.
✅ Pivot concluído: 63 linhas x 74 colunas.
✅ Arquivo salvo em 'Dados Gerados/À EXECUTAR (2026 DIANTE).xlsx'
